In [4]:
!pip -q install nibabel pydicom simpleitk scikit-learn tqdm

In [5]:

import os
import gc
import math
import json
import time
import random
import zipfile
import warnings
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
)

import nibabel as nib
import pydicom
import SimpleITK as sitk

warnings.filterwarnings("ignore")

In [6]:

from pathlib import Path
import os
import random
import numpy as np
import torch

IN_COLAB = False
IN_KAGGLE = True

WORK_BASE = Path("/kaggle/working/ablation2_gated_fusion_only")
CACHE_ROOT = WORK_BASE / "cache_mmdda_fu"
CHECKPOINT_ROOT = WORK_BASE / "checkpoints_ablation2_gated_fusion"
RESULTS_ROOT = WORK_BASE / "results_ablation2_gated_fusion"

for p in [CACHE_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CSV_ROOT = Path("/kaggle/input/datasets/syedmuabdullah99313/rp-csvs")
THREE_DATA_ROOT = Path("/kaggle/input/datasets/wajdanrasool/adni-2-pet")
ADNI1_MRI_DATA_ROOT = Path("/kaggle/input/datasets/syedmuabdullah99313/dl-project")

CSV_FILES = {
    "ADNI1_MRI": CSV_ROOT / "improvement2_adni1_mri_5_30_2026.csv",
    "ADNI1_PET": CSV_ROOT / "ADNI_1_PET_FINAL_5_30_2026.csv",
    "ADNI2_MRI": CSV_ROOT / "improvement2_adni2_mri_5_30_2026 (1).csv",
    "ADNI2_PET": CSV_ROOT / "improvement2_adni2_pet_5_30_2026.csv",
}

for k, path in CSV_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"{k} CSV not found at: {path}")

EXTRACTED = {
    "ADNI1_MRI": [ADNI1_MRI_DATA_ROOT / "ADNI"],
    "ADNI1_PET": [THREE_DATA_ROOT / "improvement2_adni1_pet_clean" / "ADNI"],
    "ADNI2_MRI": [THREE_DATA_ROOT / "improvement2_adni2_mri" / "ADNI"],
    "ADNI2_PET": [THREE_DATA_ROOT / "improvement2_adni2_pet_clean" / "ADNI"],
}

for key, roots in EXTRACTED.items():
    for root in roots:
        if not root.exists():
            raise FileNotFoundError(f"{key} image root does not exist: {root}")

print("CSV_ROOT:", CSV_ROOT)
print("THREE_DATA_ROOT:", THREE_DATA_ROOT)
print("ADNI1_MRI_DATA_ROOT:", ADNI1_MRI_DATA_ROOT)

print("\nCSV files:")
for k, v in CSV_FILES.items():
    print(k, "->", v)

print("\nImage roots:")
for k, v in EXTRACTED.items():
    print(k, "->", v)

SEED = 42
TARGET_SHAPE = (113, 137, 113)
MAX_PAIR_GAP_DAYS = 180
TASK_NAME = "AD_vs_CN"

BATCH_SIZE = 1
LR = 9e-7
DROPOUT = 0.5
TRAIN_EPOCHS = 150

CHECKPOINT_MONITOR = "AUC"
USE_SOURCE_VAL_THRESHOLD_TUNING = True
THRESHOLD_METRIC = "balanced_accuracy"

PREFILTER_BEFORE_SPLIT = True
CLEAR_CACHE = False
PREFER_PREPROCESSED_CANDIDATES = True
RAW_APPROX_FOREGROUND_CROP = True
RAW_APPROX_GAUSSIAN_SIGMA = 1.0
RAW_APPROX_USE_N4_BIAS_CORRECTION = False

PAPER_AD_CN_TOTAL = 279 + 180
MIN_RELIABLE_SOURCE_SIZE = 50

USE_GATED_FUSION = True
USE_FOCAL_LOSS = False
USE_DOMAIN_ADAPTATION = False


def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nWORK_BASE:", WORK_BASE)
print("CACHE_ROOT:", CACHE_ROOT)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("DEVICE:", DEVICE)
print("\nAblation switches:")
print("  USE_GATED_FUSION:", USE_GATED_FUSION)
print("  USE_FOCAL_LOSS:", USE_FOCAL_LOSS)
print("  USE_DOMAIN_ADAPTATION:", USE_DOMAIN_ADAPTATION)


CSV_ROOT: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs
THREE_DATA_ROOT: /kaggle/input/datasets/wajdanrasool/adni-2-pet
ADNI1_MRI_DATA_ROOT: /kaggle/input/datasets/syedmuabdullah99313/dl-project

CSV files:
ADNI1_MRI -> /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni1_mri_5_30_2026.csv
ADNI1_PET -> /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/ADNI_1_PET_FINAL_5_30_2026.csv
ADNI2_MRI -> /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni2_mri_5_30_2026 (1).csv
ADNI2_PET -> /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni2_pet_5_30_2026.csv

Image roots:
ADNI1_MRI -> [PosixPath('/kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI')]
ADNI1_PET -> [PosixPath('/kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni1_pet_clean/ADNI')]
ADNI2_MRI -> [PosixPath('/kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_mri/ADNI')]
ADNI2_PET -> [PosixPath('/kaggle/input/datasets/wajdanrasool/adni-2-pet/i

In [7]:

print("\nDetailed folder check:")
for key, roots in EXTRACTED.items():
    print("\n" + key)
    root = roots[0]
    print("  selected root:", root)
    children = list(root.iterdir())
    dirs = [x for x in children if x.is_dir()]
    files = [x for x in children if x.is_file()]
    print("  child dirs:", len(dirs))
    print("  child files:", len(files))
    print("  first dirs:", [d.name for d in dirs[:10]])
    print("  first files:", [f.name for f in files[:10]])



Detailed folder check:

ADNI1_MRI
  selected root: /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI
  child dirs: 396
  child files: 0
  first dirs: ['133_S_0913', '094_S_1090', '128_S_0272', '027_S_1387', '031_S_0618', '027_S_1277', '137_S_0669', '005_S_0221', '002_S_0685', '094_S_0531']
  first files: []

ADNI1_PET
  selected root: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni1_pet_clean/ADNI
  child dirs: 85
  child files: 0
  first dirs: ['094_S_1090', '128_S_0272', '031_S_0618', '005_S_0221', '137_S_0841', '128_S_0245', '005_S_0929', '005_S_0610', '131_S_0497', '007_S_1339']
  first files: []

ADNI2_MRI
  selected root: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_mri/ADNI
  child dirs: 399
  child files: 0
  first dirs: ['082_S_5029', '067_S_0257', '022_S_4266', '128_S_4599', '037_S_5126', '002_S_0685', '032_S_0214', '116_S_4092', '011_S_4105', '136_S_4269']
  first files: []

ADNI2_PET
  selected root: /kaggle/input/datasets/waj

In [8]:

for key, path in CSV_FILES.items():
    df = pd.read_csv(path)
    print("\n", key)
    print("path:", path)
    print("rows:", len(df))
    print("unique subjects:", df["Subject"].nunique())
    print(df["Group"].value_counts())



 ADNI1_MRI
path: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni1_mri_5_30_2026.csv
rows: 1862
unique subjects: 400
Group
MCI    1005
CN      535
AD      322
Name: count, dtype: int64

 ADNI1_PET
path: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/ADNI_1_PET_FINAL_5_30_2026.csv
rows: 406
unique subjects: 88
Group
CN    232
AD    174
Name: count, dtype: int64

 ADNI2_MRI
path: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni2_mri_5_30_2026 (1).csv
rows: 1547
unique subjects: 412
Group
CN     905
AD     344
MCI    298
Name: count, dtype: int64

 ADNI2_PET
path: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni2_pet_5_30_2026.csv
rows: 747
unique subjects: 549
Group
CN     428
AD     172
MCI    147
Name: count, dtype: int64


In [9]:

adni1_mri_csv = pd.read_csv(CSV_FILES["ADNI1_MRI"])
adni1_mri_root = EXTRACTED["ADNI1_MRI"][0]

csv_subjects = set(adni1_mri_csv["Subject"].astype(str))
folder_subjects = set([p.name for p in adni1_mri_root.iterdir() if p.is_dir()])

print("ADNI1 MRI CSV unique subjects:", len(csv_subjects))
print("ADNI1 MRI folder subject dirs:", len(folder_subjects))
print("Subjects in both:", len(csv_subjects & folder_subjects))
print("CSV subjects missing from folders:", len(csv_subjects - folder_subjects))
print("Folder subjects not in CSV:", len(folder_subjects - csv_subjects))
print("\nExample matched subjects:")
print(sorted(list(csv_subjects & folder_subjects))[:20])


ADNI1 MRI CSV unique subjects: 400
ADNI1 MRI folder subject dirs: 396
Subjects in both: 396
CSV subjects missing from folders: 4
Folder subjects not in CSV: 0

Example matched subjects:
['002_S_0295', '002_S_0413', '002_S_0559', '002_S_0619', '002_S_0685', '002_S_0729', '002_S_0782', '002_S_0816', '002_S_0938', '002_S_0954', '002_S_0955', '002_S_1018', '002_S_1070', '002_S_1155', '002_S_1261', '002_S_1268', '002_S_1280', '005_S_0221', '005_S_0222', '005_S_0223']


In [10]:
SANITY_MODE = False

if SANITY_MODE:
    TRAIN_EPOCHS = 2
    print("SANITY_MODE is ON -> using tiny epoch count for debugging.")
else:
    print("SANITY_MODE is OFF -> using full epoch count.")

print("TRAIN_EPOCHS:", TRAIN_EPOCHS)


SANITY_MODE is OFF -> using full epoch count.
TRAIN_EPOCHS: 150


In [11]:
def load_csv(path: Path, modality_name: str, dataset_name: str):
    if not path.exists():
        raise FileNotFoundError(f"Missing CSV: {path}")

    df = pd.read_csv(path).copy()
    rename_map = {
        "Image Data ID": "image_id",
        "Subject": "subject",
        "Group": "group",
        "Sex": "sex",
        "Age": "age",
        "Visit": "visit",
        "Modality": "modality",
        "Description": "description",
        "Type": "type",
        "Acq Date": "acq_date",
        "Format": "format",
        "Downloaded": "downloaded",
    }
    df = df.rename(columns=rename_map)
    df["dataset"] = dataset_name
    df["modality_name"] = modality_name
    df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")
    df["subject"] = df["subject"].astype(str)
    df["visit"] = df["visit"].astype(str)
    df["group"] = df["group"].astype(str).str.upper()
    df["label_num"] = df["group"].map({"CN": 0, "AD": 1})
    df = df[df["group"].isin(["CN", "AD"])].copy()
    return df

adni1_mri = load_csv(CSV_FILES["ADNI1_MRI"], "MRI", "ADNI1")
adni1_pet = load_csv(CSV_FILES["ADNI1_PET"], "PET", "ADNI1")
adni2_mri = load_csv(CSV_FILES["ADNI2_MRI"], "MRI", "ADNI2")
adni2_pet = load_csv(CSV_FILES["ADNI2_PET"], "PET", "ADNI2")

adni1_pet = adni1_pet[
    ~adni1_pet["description"]
    .astype(str)
    .str.upper()
    .str.contains("PIB|C11|C-11", regex=True, na=False)
].copy()

print("ADNI1 PET after removing PIB/C11 rows:", adni1_pet.shape)
print("ADNI1 PET group counts:", adni1_pet["group"].value_counts().to_dict())
print("ADNI1 PET unique subjects:", adni1_pet["subject"].nunique())


for name, df in [
    ("ADNI1 MRI", adni1_mri),
    ("ADNI1 PET", adni1_pet),
    ("ADNI2 MRI", adni2_mri),
    ("ADNI2 PET", adni2_pet),
]:
    print(f"\n{name}: {df.shape}")
    print(df[['subject', 'group', 'visit', 'image_id', 'acq_date']].head(3))
    print(df['group'].value_counts().to_dict())

ADNI1 PET after removing PIB/C11 rows: (351, 15)
ADNI1 PET group counts: {'CN': 203, 'AD': 148}
ADNI1 PET unique subjects: 88

ADNI1 MRI: (857, 15)
       subject group visit image_id   acq_date
8   137_S_1041    AD    sc   I29268 2006-11-09
9   137_S_1041    AD   m06   I56100 2007-06-06
10  137_S_1041    AD   m12   I84667 2007-12-12
{'CN': 535, 'AD': 322}

ADNI1 PET: (351, 15)
      subject group visit image_id   acq_date
0  137_S_1041    AD    bl   I32968 2006-12-12
1  137_S_1041    AD   m06   I55363 2007-05-29
2  137_S_1041    AD   m12   I85759 2007-12-18
{'CN': 203, 'AD': 148}

ADNI2 MRI: (1249, 15)
      subject group visit image_id   acq_date
0  941_S_4376    CN   v02  I269043 2011-06-01
1  941_S_4376    CN   v02  I276860 2012-01-06
2  941_S_4376    CN   v04  I294442 2012-03-30
{'CN': 905, 'AD': 344}

ADNI2 PET: (600, 15)
      subject group visit image_id   acq_date
0  941_S_4376    CN   v03  I283469 2012-02-08
1  941_S_4365    CN   v03  I274743 2011-12-30
2  941_S_4365    CN   

In [12]:
def build_nearest_pairs(mri_df, pet_df, dataset_name, max_gap_days=180):
    shared_subjects = sorted(set(mri_df["subject"]) & set(pet_df["subject"]))
    rows = []

    for subj in shared_subjects:
        msub = mri_df[mri_df["subject"] == subj].copy()
        psub = pet_df[pet_df["subject"] == subj].copy()

        best = None
        best_key = None

        for _, mr in msub.iterrows():
            for _, pr in psub.iterrows():
                same_group = int(mr["group"] == pr["group"])
                if pd.isna(mr["acq_date"]) or pd.isna(pr["acq_date"]):
                    gap_days = 999999
                else:
                    gap_days = abs((mr["acq_date"] - pr["acq_date"]).days)


                score = (-same_group, gap_days)

                if best is None or score < best_key:
                    best = (mr, pr)
                    best_key = score

        if best is None:
            continue

        mr, pr = best
        if mr["group"] != pr["group"]:
            continue

        if pd.isna(mr["acq_date"]) or pd.isna(pr["acq_date"]):
            continue

        gap_days = abs((mr["acq_date"] - pr["acq_date"]).days)
        if gap_days > max_gap_days:
            continue

        rows.append({
            "dataset": dataset_name,
            "subject": subj,
            "group": mr["group"],
            "label_num": int(mr["label_num"]),
            "sex": mr["sex"],
            "age": float(mr["age"]),
            "mri_visit": mr["visit"],
            "pet_visit": pr["visit"],
            "mri_image_id": str(mr["image_id"]),
            "pet_image_id": str(pr["image_id"]),
            "mri_date": mr["acq_date"],
            "pet_date": pr["acq_date"],
            "pair_gap_days": int(gap_days),
        })

    out = pd.DataFrame(rows).sort_values(["dataset", "subject"]).reset_index(drop=True)
    return out

paired_adni1 = build_nearest_pairs(adni1_mri, adni1_pet, "ADNI1", max_gap_days=MAX_PAIR_GAP_DAYS)
paired_adni2 = build_nearest_pairs(adni2_mri, adni2_pet, "ADNI2", max_gap_days=MAX_PAIR_GAP_DAYS)

print("paired_adni1:", paired_adni1.shape)
print(paired_adni1["group"].value_counts().to_dict())
print(paired_adni1["pair_gap_days"].describe())

print("\npaired_adni2:", paired_adni2.shape)
print(paired_adni2["group"].value_counts().to_dict())
print(paired_adni2["pair_gap_days"].describe())

display(paired_adni1.head())
display(paired_adni2.head())

paired_adni1: (87, 13)
{'CN': 45, 'AD': 42}
count    87.000000
mean      8.275862
std      14.543191
min       0.000000
25%       0.000000
50%       1.000000
75%       9.500000
max      65.000000
Name: pair_gap_days, dtype: float64

paired_adni2: (281, 13)
{'CN': 181, 'AD': 100}
count    281.000000
mean      19.843416
std       24.948168
min        0.000000
25%        4.000000
50%       13.000000
75%       26.000000
max      159.000000
Name: pair_gap_days, dtype: float64


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days
0,ADNI1,005_S_0221,AD,1,M,68.0,m06,m06,I25942,I25918,2006-10-06,2006-10-06,0
1,ADNI1,005_S_0223,CN,0,F,79.0,m06,m06,I26116,I26108,2006-10-11,2006-10-11,0
2,ADNI1,005_S_0610,CN,0,M,80.0,m06,m06,I39068,I39087,2007-02-09,2007-02-09,0
3,ADNI1,005_S_0929,AD,1,M,83.0,m06,m06,I53858,I57389,2007-05-09,2007-06-19,41
4,ADNI1,005_S_1341,AD,1,F,72.0,m06,m06,I76567,I76770,2007-10-02,2007-10-02,0


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days
0,ADNI2,002_S_0295,CN,0,M,90.0,v06,v06,I238627,I239487,2011-06-02,2011-06-09,7
1,ADNI2,002_S_0413,CN,0,F,82.0,v06,v06,I240812,I240813,2011-06-16,2011-06-17,1
2,ADNI2,002_S_0685,CN,0,F,96.0,v11,v11,I322437,I321228,2012-07-27,2012-08-02,6
3,ADNI2,002_S_1261,CN,0,F,77.0,v11,v11,I361610,I363184,2013-02-27,2013-03-07,8
4,ADNI2,002_S_1280,CN,0,F,77.0,v11,v11,I361326,I360640,2013-02-26,2013-02-19,7


In [13]:
from pathlib import Path
from tqdm.auto import tqdm

print("Building path indexes using selected ADNI roots only. This may take time but avoids duplicate parent-folder scans.")

VOLUME_EXTENSIONS = (
    ".nii", ".nii.gz", ".img", ".hdr", ".v", ".mnc", ".nrrd", ".mha", ".mhd", ".mgz"
)

def build_path_index(search_roots):
    index = {
        "dirs": [],
        "vol_files": [],
    }

    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue

        print(f"Scanning root: {root}")

        count = 0
        for p in root.rglob("*"):
            count += 1
            if count % 50000 == 0:
                print(f"  scanned {count:,} paths... dirs={len(index['dirs']):,}, vol_files={len(index['vol_files']):,}")

            low = str(p).lower()
            if p.is_dir():
                index["dirs"].append((low, p))
            elif p.is_file() and low.endswith(VOLUME_EXTENSIONS):
                index["vol_files"].append((low, p))

        print(f"Finished root: {root}")
        print(f"  total scanned: {count:,}")
        print(f"  dirs: {len(index['dirs']):,}")
        print(f"  volume files: {len(index['vol_files']):,}")

    return index

INDEXES = {
    "ADNI1_MRI": build_path_index(EXTRACTED["ADNI1_MRI"]),
    "ADNI1_PET": build_path_index(EXTRACTED["ADNI1_PET"]),
    "ADNI2_MRI": build_path_index(EXTRACTED["ADNI2_MRI"]),
    "ADNI2_PET": build_path_index(EXTRACTED["ADNI2_PET"]),
}

for k, idx in INDEXES.items():
    print(k, "dirs:", len(idx["dirs"]), "| vol files:", len(idx["vol_files"]))

Building path indexes using selected ADNI roots only. This may take time but avoids duplicate parent-folder scans.
Scanning root: /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI
  scanned 50,000 paths... dirs=1,370, vol_files=0
  scanned 100,000 paths... dirs=1,950, vol_files=0
  scanned 150,000 paths... dirs=2,550, vol_files=0
  scanned 200,000 paths... dirs=3,114, vol_files=0
  scanned 250,000 paths... dirs=3,690, vol_files=0
Finished root: /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI
  total scanned: 273,845
  dirs: 3,965
  volume files: 0
Scanning root: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni1_pet_clean/ADNI
  scanned 50,000 paths... dirs=1,065, vol_files=564
Finished root: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni1_pet_clean/ADNI
  total scanned: 50,005
  dirs: 1,067
  volume files: 568
Scanning root: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_mri/ADNI
  scanned 50,000 paths... dirs=1,3

In [14]:

import shutil

if CLEAR_CACHE:
    shutil.rmtree(CACHE_ROOT, ignore_errors=True)
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Cache cleared.")
else:
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    print("CLEAR_CACHE=False -> keeping/reusing existing cache if present.")
    print("Existing cached npy files:", len(list(CACHE_ROOT.glob("*.npy"))))


CLEAR_CACHE=False -> keeping/reusing existing cache if present.
Existing cached npy files: 0


In [15]:
def locate_study_path(index, image_id, subject, visit=None):
    image_id = str(image_id).lower()
    subject = str(subject).lower()
    visit = None if visit is None else str(visit).lower()

    def is_v_file(p):
        return str(p).lower().endswith(".v")

    hit_dirs = [p for low, p in index["dirs"] if image_id in low]
    if hit_dirs:
        return sorted(hit_dirs, key=lambda x: len(str(x)), reverse=True)[0]

    hit_files = [p for low, p in index["vol_files"] if image_id in low and not is_v_file(p)]
    if hit_files:
        return sorted(hit_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    hit_v_files = [p for low, p in index["vol_files"] if image_id in low and is_v_file(p)]
    if hit_v_files:
        return sorted(hit_v_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    subject_dirs = []
    for low, p in index["dirs"]:
        if subject in low and (visit is None or visit in low):
            subject_dirs.append(p)

    if subject_dirs:
        return sorted(subject_dirs, key=lambda x: len(str(x)), reverse=True)[0]

    subject_dirs = [p for low, p in index["dirs"] if subject in low]
    if subject_dirs:
        return sorted(subject_dirs, key=lambda x: len(str(x)), reverse=True)[0]


    subject_files = []
    for low, p in index["vol_files"]:
        if subject in low and (visit is None or visit in low) and not is_v_file(p):
            subject_files.append(p)

    if subject_files:
        return sorted(subject_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]


    subject_v_files = []
    for low, p in index["vol_files"]:
        if subject in low and (visit is None or visit in low) and is_v_file(p):
            subject_v_files.append(p)

    if subject_v_files:
        return sorted(subject_v_files, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[0]

    raise FileNotFoundError(
        f"Could not locate study path for image_id={image_id}, subject={subject}, visit={visit}"
    )

In [16]:
PAPER_PREPROCESSED_KEYWORDS = (
    "cat12", "spm", "mni", "smwc", "mwp", "warped", "normalized", "normalised",
    "registered", "coreg", "brain", "strip", "skull", "resliced", "resample",
    "smooth", "wc1", "wc2", "mrireg", "petreg"
)

def _is_probably_preprocessed_path(path: Path):
    low = str(path).lower()
    return any(k in low for k in PAPER_PREPROCESSED_KEYWORDS)


def _ensure_3d_volume(arr, source):
    arr = np.asarray(arr, dtype=np.float32)
    while arr.ndim > 3:
        arr = arr[..., 0]
    if arr.ndim != 3:
        raise RuntimeError(f"Expected 3D volume from {source}, got shape {arr.shape}")
    return arr.astype(np.float32)


def _load_analyze_like_hdr_pair(hdr_path: Path):
    hdr_path = Path(hdr_path)
    low = str(hdr_path).lower()

    if not low.endswith(".hdr"):
        raise RuntimeError(f"Not a .hdr file: {hdr_path}")

    candidates = [hdr_path.with_suffix(".img")]

    if low.endswith(".i.hdr"):
        candidates.append(Path(str(hdr_path)[:-4]))

    candidates.append(Path(str(hdr_path)[:-4]))

    seen = set()
    unique_candidates = []
    for c in candidates:
        s = str(c)
        if s not in seen:
            seen.add(s)
            unique_candidates.append(c)

    data_path = None
    for c in unique_candidates:
        if c.exists() and c.is_file():
            data_path = c
            break

    if data_path is None:
        raise RuntimeError(f"Could not find paired binary file for header: {hdr_path}")

    try:
        from nibabel.analyze import AnalyzeHeader

        with open(hdr_path, "rb") as f_hdr:
            hdr = AnalyzeHeader.from_fileobj(f_hdr)

        with open(data_path, "rb") as f_img:
            arr = hdr.data_from_fileobj(f_img)

        return _ensure_3d_volume(arr, f"{hdr_path} + {data_path}")
    except Exception as e:
        raise RuntimeError(f"Failed to load Analyze-like pair: {hdr_path} + {data_path}") from e


def load_single_volume_file(path: Path):
    path = Path(path)
    low = str(path).lower()

    if low.endswith(".v"):
        try:
            from nibabel import ecat
            img = ecat.load(str(path))
            arr = np.asarray(img.get_fdata(), dtype=np.float32)
            return _ensure_3d_volume(arr, path)
        except Exception:
            pass

    if low.endswith(".hdr"):
        try:
            return _load_analyze_like_hdr_pair(path)
        except Exception:
            pass

    try:
        img = nib.load(str(path))
        arr = np.asarray(img.get_fdata(), dtype=np.float32)
        return _ensure_3d_volume(arr, path)
    except Exception:
        pass

    try:
        img = sitk.ReadImage(str(path))
        arr = sitk.GetArrayFromImage(img).astype(np.float32)
        arr = np.transpose(arr, (2, 1, 0))
        return _ensure_3d_volume(arr, path)
    except Exception as e:
        raise RuntimeError(f"Could not load single-volume file: {path}") from e


def _dicom_sort_value(ds):
    instance_no = getattr(ds, "InstanceNumber", None)
    if instance_no is not None:
        try:
            return float(instance_no)
        except Exception:
            pass

    ipp = getattr(ds, "ImagePositionPatient", None)
    if ipp is not None and len(ipp) >= 3:
        try:
            return float(ipp[-1])
        except Exception:
            pass

    slice_location = getattr(ds, "SliceLocation", None)
    if slice_location is not None:
        try:
            return float(slice_location)
        except Exception:
            pass

    return 0.0


def load_dicom_series_from_dir(series_dir: Path):
    series_dir = Path(series_dir)

    try:
        reader = sitk.ImageSeriesReader()
        series_ids = reader.GetGDCMSeriesIDs(str(series_dir))
        if series_ids:
            series_candidates = []
            for sid in series_ids:
                files = list(reader.GetGDCMSeriesFileNames(str(series_dir), sid))
                series_candidates.append((len(files), sid, files))

            for _, sid, files in sorted(series_candidates, reverse=True):
                try:
                    reader.SetFileNames(files)
                    img = reader.Execute()
                    arr = sitk.GetArrayFromImage(img).astype(np.float32)
                    arr = np.transpose(arr, (2, 1, 0))
                    return _ensure_3d_volume(arr, f"{series_dir}::{sid}")
                except Exception:
                    continue
    except Exception:
        pass

    grouped_slices = {}
    multiframe_volumes = []

    for p in series_dir.rglob("*"):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=False, force=True)
            if not hasattr(ds, "pixel_array"):
                continue
            arr = np.asarray(ds.pixel_array, dtype=np.float32)
        except Exception:
            continue

        series_uid = str(getattr(ds, "SeriesInstanceUID", "unknown"))

        if arr.ndim == 3:
            vol = np.transpose(arr, (1, 2, 0))
            multiframe_volumes.append((vol.size, vol))
            continue

        if arr.ndim != 2:
            continue

        rows = int(getattr(ds, "Rows", arr.shape[0]))
        cols = int(getattr(ds, "Columns", arr.shape[1]))
        key = (series_uid, rows, cols)
        grouped_slices.setdefault(key, []).append((_dicom_sort_value(ds), arr))

    volume_candidates = []

    for _, vol in multiframe_volumes:
        try:
            volume_candidates.append((vol.size, _ensure_3d_volume(vol, series_dir)))
        except Exception:
            continue

    for key, slices in grouped_slices.items():
        shapes = {sl.shape for _, sl in slices}
        if len(shapes) != 1 or len(slices) == 0:
            continue
        slices = sorted(slices, key=lambda x: x[0])
        try:
            vol = np.stack([x[1] for x in slices], axis=-1).astype(np.float32)
            volume_candidates.append((vol.size, _ensure_3d_volume(vol, f"{series_dir}::{key[0]}")))
        except Exception:
            continue

    if not volume_candidates:
        raise RuntimeError(f"No readable DICOM slices found inside {series_dir}")

    volume_candidates = sorted(volume_candidates, key=lambda x: x[0], reverse=True)
    return volume_candidates[0][1]


def _rank_volume_candidate(p: Path):
    low = str(p).lower()
    size = p.stat().st_size if p.exists() else 0

    preferred_bonus = -20 if (PREFER_PREPROCESSED_CANDIDATES and _is_probably_preprocessed_path(p)) else 0

    if low.endswith(".nii.gz"):
        rank = 0
    elif low.endswith(".nii"):
        rank = 1
    elif low.endswith(".mgz"):
        rank = 2
    elif low.endswith(".mha") or low.endswith(".mhd"):
        rank = 3
    elif low.endswith(".img"):
        rank = 4
    elif low.endswith(".hdr"):
        rank = 5
    elif low.endswith(".mnc"):
        rank = 6
    elif low.endswith(".nrrd"):
        rank = 7
    elif low.endswith(".v"):
        rank = 99
    else:
        rank = 50

    return (preferred_bonus + rank, -size)


def load_volume(path: Path):
    path = Path(path)

    if path.is_file():
        return load_single_volume_file(path)

    try:
        return load_dicom_series_from_dir(path)
    except Exception:
        pass

    volume_files = []
    for p in path.rglob("*"):
        if p.is_file():
            low = str(p).lower()
            if low.endswith(VOLUME_EXTENSIONS):
                volume_files.append(p)

    if not volume_files:
        raise RuntimeError(f"No readable volume found inside directory: {path}")

    volume_files = sorted(volume_files, key=_rank_volume_candidate)

    last_error = None
    for vf in volume_files:
        try:
            return load_single_volume_file(vf)
        except Exception as e:
            last_error = e
            continue

    raise RuntimeError(f"All candidate volume loaders failed for {path}") from last_error


In [17]:
def _ensure_3d_float(volume):
    v = np.asarray(volume, dtype=np.float32)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)

    while v.ndim > 3:
        v = v[..., 0]

    if v.ndim != 3:
        raise ValueError(f"Expected 3D volume after squeezing, got shape {v.shape}")

    return v.astype(np.float32)


def robust_clip(volume, lower_q=0.5, upper_q=99.5):
    v = np.asarray(volume, dtype=np.float32)
    finite_mask = np.isfinite(v)
    if finite_mask.sum() == 0:
        return np.zeros_like(v, dtype=np.float32)
    lo, hi = np.percentile(v[finite_mask], [lower_q, upper_q])
    v = np.clip(v, lo, hi)
    return v.astype(np.float32)


def crop_to_foreground(volume, modality, margin=4):
    v = _ensure_3d_float(volume)
    nonzero = np.abs(v) > 0
    if nonzero.any():
        foreground_values = v[nonzero]
    else:
        foreground_values = v[np.isfinite(v)]

    if foreground_values.size == 0:
        return v

    if modality.upper() == "MRI":
        threshold = np.percentile(foreground_values, 35)
    else:
        threshold = np.percentile(foreground_values, 55)

    mask = (v > threshold) | nonzero
    coords = np.argwhere(mask)
    if coords.size == 0:
        return v

    lo = np.maximum(coords.min(axis=0) - margin, 0)
    hi = np.minimum(coords.max(axis=0) + margin + 1, np.array(v.shape))
    cropped = v[
        int(lo[0]):int(hi[0]),
        int(lo[1]):int(hi[1]),
        int(lo[2]):int(hi[2]),
    ]
    return cropped.astype(np.float32) if cropped.size > 0 else v


def maybe_n4_bias_correct(volume):
    v = _ensure_3d_float(volume)
    if not RAW_APPROX_USE_N4_BIAS_CORRECTION:
        return v

    try:
        itk = sitk.GetImageFromArray(np.transpose(v, (2, 1, 0)))
        mask = sitk.OtsuThreshold(itk, 0, 1, 128)
        corrected = sitk.N4BiasFieldCorrection(itk, mask)
        out = sitk.GetArrayFromImage(corrected).astype(np.float32)
        return np.transpose(out, (2, 1, 0))
    except Exception:
        return v


def maybe_gaussian_smooth(volume, sigma):
    v = _ensure_3d_float(volume)
    if sigma is None or sigma <= 0:
        return v

    try:
        itk = sitk.GetImageFromArray(np.transpose(v, (2, 1, 0)))
        smoothed = sitk.SmoothingRecursiveGaussian(itk, sigma=float(sigma))
        out = sitk.GetArrayFromImage(smoothed).astype(np.float32)
        return np.transpose(out, (2, 1, 0))
    except Exception:
        return v


def normalize_mri(volume):
    v = robust_clip(volume)
    mask = v > np.percentile(v, 20)
    if mask.sum() == 0:
        mean = v.mean()
        std = v.std() + 1e-6
    else:
        mean = v[mask].mean()
        std = v[mask].std() + 1e-6
    v = (v - mean) / std
    return v.astype(np.float32)


def normalize_pet(volume):
    v = robust_clip(volume)
    v = v - v.min()
    denom = v.max() + 1e-6
    v = v / denom
    return v.astype(np.float32)


def resize_volume(volume, target_shape=TARGET_SHAPE):
    x = torch.tensor(volume, dtype=torch.float32).unsqueeze(0).unsqueeze(0) 
    x = F.interpolate(x, size=target_shape, mode="trilinear", align_corners=False)
    return x.squeeze(0).squeeze(0).cpu().numpy().astype(np.float32)


def preprocess_volume(volume, modality, source_path=None):
    v = _ensure_3d_float(volume)
    source_path = None if source_path is None else Path(source_path)
    paper_like_input = (
        source_path is not None
        and PREFER_PREPROCESSED_CANDIDATES
        and _is_probably_preprocessed_path(source_path)
    )

    if not paper_like_input and RAW_APPROX_FOREGROUND_CROP:
        v = crop_to_foreground(v, modality=modality, margin=4)

    if modality.upper() == "MRI":
        if not paper_like_input:
            v = maybe_n4_bias_correct(v)
            v = maybe_gaussian_smooth(v, sigma=RAW_APPROX_GAUSSIAN_SIGMA)
        v = normalize_mri(v)
    elif modality.upper() == "PET":
        if not paper_like_input:
            v = maybe_gaussian_smooth(v, sigma=RAW_APPROX_GAUSSIAN_SIGMA)
        v = normalize_pet(v)
    else:
        raise ValueError(f"Unknown modality: {modality}")

    if tuple(v.shape) != tuple(TARGET_SHAPE):
        v = resize_volume(v, target_shape=TARGET_SHAPE)

    if modality.upper() == "MRI":
        v = normalize_mri(v)
    else:
        v = normalize_pet(v)

    return v.astype(np.float32)


In [18]:
def cache_key(row, modality):
    image_id = row["mri_image_id"] if modality == "MRI" else row["pet_image_id"]
    return f"{row['dataset']}__{row['subject']}__{modality}__{image_id}.npy"


def get_cached_or_build(row, modality, index_key):
    out_path = CACHE_ROOT / cache_key(row, modality)

    if out_path.exists():
        try:
            arr = np.load(out_path)
            return arr.astype(np.float32)
        except Exception:
            try:
                out_path.unlink()
            except Exception:
                pass

    if modality == "MRI":
        image_id = row["mri_image_id"]
        visit = row["mri_visit"]
    else:
        image_id = row["pet_image_id"]
        visit = row["pet_visit"]

    index = INDEXES[index_key]

    try:
        source_path = locate_study_path(
            index,
            image_id=image_id,
            subject=row["subject"],
            visit=visit,
        )
        volume = load_volume(source_path)
        volume = preprocess_volume(volume, modality=modality, source_path=source_path)
        volume = volume.astype(np.float32)
        np.save(out_path, volume)
        return volume
    except Exception:
        try:
            if out_path.exists():
                out_path.unlink()
        except Exception:
            pass
        raise


In [19]:
import signal
from tqdm.auto import tqdm
import pandas as pd

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException("Image loading timed out")


def filter_loadable_pairs(df, name="dataset", timeout_sec=60):
    """
    Filters readable MRI-PET pairs using the notebook's real loading pipeline.
    It does NOT look for mri_path/pet_path columns.
    
    It uses:
      dataset
      mri_image_id
      pet_image_id
      get_cached_or_build(...)
    
    Returns:
      kept_df, bad_df
    """

    df = df.reset_index(drop=True).copy()

    kept_rows = []
    bad_rows = []

    signal.signal(signal.SIGALRM, timeout_handler)

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"filter {name}"):

        dataset = row["dataset"]

        mri_index_key = f"{dataset}_MRI"
        pet_index_key = f"{dataset}_PET"

        try:
            signal.alarm(timeout_sec)

            _ = get_cached_or_build(row, modality="MRI", index_key=mri_index_key)

            signal.alarm(timeout_sec)

            _ = get_cached_or_build(row, modality="PET", index_key=pet_index_key)

            signal.alarm(0)

            kept_rows.append(row)

        except TimeoutException:
            signal.alarm(0)
            bad_row = row.copy()
            bad_row["drop_reason"] = "timeout"
            bad_rows.append(bad_row)

        except Exception as e:
            signal.alarm(0)
            bad_row = row.copy()
            bad_row["drop_reason"] = str(e)[:300]
            bad_rows.append(bad_row)

    kept_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    bad_df = pd.DataFrame(bad_rows).reset_index(drop=True)

    print(f"{name}: kept {len(kept_df)} / {len(df)}")
    print(f"{name}: dropped {len(bad_df)} unreadable or timeout pairs")

    return kept_df, bad_df

In [20]:

class PairedADNIDataset(Dataset):
    def __init__(self, df, split_name):
        self.df = df.reset_index(drop=True).copy()
        self.split_name = split_name

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()

        dataset = row["dataset"]
        mri_index_key = f"{dataset}_MRI"
        pet_index_key = f"{dataset}_PET"

        mri = get_cached_or_build(row, modality="MRI", index_key=mri_index_key)
        pet = get_cached_or_build(row, modality="PET", index_key=pet_index_key)

        mri = torch.tensor(mri, dtype=torch.float32).unsqueeze(0)
        pet = torch.tensor(pet, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(int(row["label_num"]), dtype=torch.long)

        return {
            "mri": mri,
            "pet": pet,
            "label": label,
            "subject": row["subject"],
            "dataset": row["dataset"],
        }

def make_loader(df, split_name, shuffle):
    ds = PairedADNIDataset(df, split_name=split_name)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

In [21]:
def make_domain_splits(df, seed=42):
    df = df.reset_index(drop=True).copy()

    train_val_df, test_df = train_test_split(
        df,
        test_size=0.30,
        random_state=seed,
        stratify=df["label_num"],
    )

    val_ratio_within_train = 0.30 / 0.70
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_ratio_within_train,
        random_state=seed,
        stratify=train_val_df["label_num"],
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


if PREFILTER_BEFORE_SPLIT:
    print(
        "Filtering readable MRI/PET pairs before splitting so the train/val/test split is made "
        "on the final usable sample set."
    )
    paired_adni1_f, paired_adni1_bad = filter_loadable_pairs(paired_adni1, "paired_adni1")
    paired_adni2_f, paired_adni2_bad = filter_loadable_pairs(paired_adni2, "paired_adni2")
    dropped_all = pd.concat([paired_adni1_bad, paired_adni2_bad], ignore_index=True)
else:
    paired_adni1_f, paired_adni1_bad = paired_adni1.copy(), pd.DataFrame()
    paired_adni2_f, paired_adni2_bad = paired_adni2.copy(), pd.DataFrame()
    dropped_all = pd.DataFrame()

total_usable = len(paired_adni1_f) + len(paired_adni2_f)
print(f"Usable AD/CN pairs in this run: {total_usable}")
print(f"Paper AD/CN total subjects    : {PAPER_AD_CN_TOTAL}")

if total_usable < PAPER_AD_CN_TOTAL:
    print(
        "[warning] Current AD/CN coverage is smaller than the paper's. Matching the paper's accuracy "
        "is unlikely unless the missing subjects and paper-style preprocessing are restored."
    )

if len(paired_adni1_f) < MIN_RELIABLE_SOURCE_SIZE:
    print(
        f"[warning] ADNI1 usable pairs = {len(paired_adni1_f)}. "
        "After the paper-style split, ADNI1->ADNI2 source training will be very data-limited."
    )

adni1_train, adni1_val, adni1_test = make_domain_splits(paired_adni1_f, seed=SEED)
adni2_train, adni2_val, adni2_test = make_domain_splits(paired_adni2_f, seed=SEED)

adni1_train_f, adni1_val_f, adni1_test_f = adni1_train, adni1_val, adni1_test
adni2_train_f, adni2_val_f, adni2_test_f = adni2_train, adni2_val, adni2_test

print("ADNI1 splits:")
for name, dfx in [("train", adni1_train_f), ("val", adni1_val_f), ("test", adni1_test_f)]:
    print(name, dfx.shape, dfx["group"].value_counts().to_dict())

print("\nADNI2 splits:")
for name, dfx in [("train", adni2_train_f), ("val", adni2_val_f), ("test", adni2_test_f)]:
    print(name, dfx.shape, dfx["group"].value_counts().to_dict())


Filtering readable MRI/PET pairs before splitting so the train/val/test split is made on the final usable sample set.


filter paired_adni1:   0%|          | 0/87 [00:00<?, ?it/s]

GDCMSeriesFileNames (0x159d25b0): No Series were found

GDCMSeriesFileNames (0x159d25b0): No Series were found

ImageSeriesReader (0x159d25b0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.0002

GDCMSeriesFileNames (0x159d25b0): No Series were found

ImageSeriesReader (0x159d25b0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000398182

GDCMSeriesFileNames (0x159d25b0): No Series were found

ImageSeriesReader (0x159d25b0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000297576

GDCMSeriesFileNames (0x159d25b0): No Series were found

ImageSeriesReader (0x159d25b0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000398182

GDCMSeriesFileNames (0x159d25b0): No Series were found

ImageSeriesReader (0x159d25b0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000496364

GDCMSeriesFileNames (0x159d25b0): No Series were found

GDCMSeriesFileNames (0x159d2

paired_adni1: kept 66 / 87
paired_adni1: dropped 21 unreadable or timeout pairs


filter paired_adni2:   0%|          | 0/281 [00:00<?, ?it/s]

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:177.67

ImageSeriesReader (0x160a38c0): Non uniform sampling or missing slices detected,  maximum nonuniformity:

paired_adni2: kept 249 / 281
paired_adni2: dropped 32 unreadable or timeout pairs
Usable AD/CN pairs in this run: 315
Paper AD/CN total subjects    : 459
[warning] Current AD/CN coverage is smaller than the paper's. Matching the paper's accuracy is unlikely unless the missing subjects and paper-style preprocessing are restored.
ADNI1 splits:
train (26, 13) {'AD': 14, 'CN': 12}
val (20, 13) {'CN': 10, 'AD': 10}
test (20, 13) {'CN': 10, 'AD': 10}

ADNI2 splits:
train (99, 13) {'CN': 65, 'AD': 34}
val (75, 13) {'CN': 50, 'AD': 25}
test (75, 13) {'CN': 50, 'AD': 25}


In [22]:
sample_a = adni1_train.iloc[0].to_dict()
mri_a = get_cached_or_build(sample_a, modality="MRI", index_key="ADNI1_MRI")
pet_a = get_cached_or_build(sample_a, modality="PET", index_key="ADNI1_PET")
print("ADNI1 MRI shape:", mri_a.shape, "PET shape:", pet_a.shape)

sample_b = adni2_train.iloc[0].to_dict()
mri_b = get_cached_or_build(sample_b, modality="MRI", index_key="ADNI2_MRI")
pet_b = get_cached_or_build(sample_b, modality="PET", index_key="ADNI2_PET")
print("ADNI2 MRI shape:", mri_b.shape, "PET shape:", pet_b.shape)

ADNI1 MRI shape: (113, 137, 113) PET shape: (113, 137, 113)
ADNI2 MRI shape: (113, 137, 113) PET shape: (113, 137, 113)


In [23]:
print("Usable split summary (filtering was performed before splitting):")
print("ADNI1 usable total:", len(paired_adni1_f))
print("ADNI2 usable total:", len(paired_adni2_f))
print("ADNI1 train/val/test:", len(adni1_train_f), len(adni1_val_f), len(adni1_test_f))
print("ADNI2 train/val/test:", len(adni2_train_f), len(adni2_val_f), len(adni2_test_f))
print("Total dropped before split:", len(dropped_all))

if len(dropped_all) > 0:
    display(dropped_all.head(20))


Usable split summary (filtering was performed before splitting):
ADNI1 usable total: 66
ADNI2 usable total: 249
ADNI1 train/val/test: 26 20 20
ADNI2 train/val/test: 99 75 75
Total dropped before split: 53


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days,drop_reason
0,ADNI1,009_S_0751,CN,0,M,74.0,m36,m36,I154019,I154497,2009-08-24,2009-09-03,10,Could not locate study path for image_id=i1540...
1,ADNI1,009_S_0842,CN,0,M,77.0,m36,m36,I156154,I156112,2009-10-05,2009-10-06,1,Could not locate study path for image_id=i1561...
2,ADNI1,009_S_0862,CN,0,F,77.0,m36,m36,I157179,I157578,2009-10-14,2009-10-20,6,Could not locate study path for image_id=i1571...
3,ADNI1,029_S_0836,AD,1,M,84.0,m12,m12,I76494,I76321,2007-10-01,2007-09-20,11,All candidate volume loaders failed for /kaggl...
4,ADNI1,029_S_0843,CN,0,M,71.0,sc,bl,I24406,I32461,2006-09-13,2006-11-17,65,All candidate volume loaders failed for /kaggl...
5,ADNI1,029_S_0845,CN,0,M,81.0,m06,m06,I47780,I48324,2007-04-02,2007-04-02,0,All candidate volume loaders failed for /kaggl...
6,ADNI1,029_S_0866,CN,0,F,81.0,m06,m06,I49181,I49780,2007-04-12,2007-04-12,0,Could not locate study path for image_id=i4918...
7,ADNI1,029_S_1056,AD,1,F,72.0,m06,m06,I57376,I58447,2007-06-19,2007-06-19,0,All candidate volume loaders failed for /kaggl...
8,ADNI1,053_S_1044,AD,1,M,66.0,m06,m06,I56740,I56047,2007-06-04,2007-06-04,0,All candidate volume loaders failed for /kaggl...
9,ADNI1,098_S_0149,AD,1,M,88.0,sc,bl,I10146,I11767,2006-01-22,2006-03-08,45,Could not locate study path for image_id=i1176...


In [24]:

class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=1, padding=0, bias=False)
        self.bn = nn.BatchNorm3d(out_ch)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Encoder3D(nn.Module):
    """Same lightweight 9-layer 3D CNN encoder family used in the proposed notebook."""
    def __init__(self):
        super().__init__()
        channels = [8, 8, 16, 16, 32, 32, 64, 64, 128]
        blocks = []
        in_ch = 1
        for out_ch in channels:
            blocks.append(ConvBlock3D(in_ch, out_ch))
            in_ch = out_ch
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        for i, block in enumerate(self.blocks, start=1):
            x = block(x)
            if i in {2, 4, 6, 8}:
                x = self.pool(x)
        return x


class GatedFusion(nn.Module):
    """
    Learnable MRI-PET fusion gate.

    The gate is an embedding-wise vector in [0, 1]. For each feature dimension:
      fused = gate * MRI_embedding + (1 - gate) * PET_embedding

    A gate value closer to 1 means that dimension relies more on MRI.
    A gate value closer to 0 means that dimension relies more on PET.
    """
    def __init__(self, in_channels=128, embed_dim=128, dropout=0.5):
        super().__init__()
        self.mri_proj = nn.Sequential(
            nn.Linear(in_channels, embed_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
        )
        self.pet_proj = nn.Sequential(
            nn.Linear(in_channels, embed_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
        )
        self.gate_net = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid(),
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, f_mri, f_pet):
        mri_vec = f_mri.mean(dim=(2, 3, 4))
        pet_vec = f_pet.mean(dim=(2, 3, 4))

        mri_emb = self.mri_proj(mri_vec)
        pet_emb = self.pet_proj(pet_vec)

        gate_in = torch.cat([mri_emb, pet_emb, torch.abs(mri_emb - pet_emb)], dim=1)
        gate = self.gate_net(gate_in)

        fused = gate * mri_emb + (1.0 - gate) * pet_emb
        fused = self.norm(fused)
        return fused, mri_emb, pet_emb, gate


class GatedCNNFusionOnly(nn.Module):
    """
    Second ablation: gated fusion only.

    MRI and PET each pass through a 3D CNN encoder. We global-average-pool the feature
    maps, project each modality to an embedding, combine them through a learnable gate,
    and classify the fused embedding.

    Removed on purpose:
      - no focal loss: this notebook uses plain cross-entropy
      - no domain discriminator / GRL
      - no unlabeled target-domain adaptation loss
    """
    def __init__(self, in_channels=128, embed_dim=128, dropout=0.5):
        super().__init__()
        self.mri_encoder = Encoder3D()
        self.pet_encoder = Encoder3D()
        self.fusion = GatedFusion(in_channels=in_channels, embed_dim=embed_dim, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def encode(self, mri, pet):
        f_mri = self.mri_encoder(mri)
        f_pet = self.pet_encoder(pet)
        fused, mri_emb, pet_emb, gate = self.fusion(f_mri, f_pet)
        return {
            "f_mri": f_mri,
            "f_pet": f_pet,
            "mri_emb": mri_emb,
            "pet_emb": pet_emb,
            "fused": fused,
            "gate": gate,
        }

    def classify(self, fused):
        return self.classifier(fused)

    def forward(self, mri, pet):
        enc = self.encode(mri, pet)
        logits = self.classify(enc["fused"])
        return logits


In [25]:

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    spe = tn / (tn + fp + 1e-8)
    sen = tp / (tp + fn + 1e-8)
    bal_acc = 0.5 * (sen + spe)

    return {
        "ACC": acc * 100.0,
        "BAL_ACC": bal_acc * 100.0,
        "SEN": sen * 100.0,
        "SPE": spe * 100.0,
        "AUC": auc * 100.0 if not np.isnan(auc) else np.nan,
        "F1": f1 * 100.0,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "threshold": float(threshold),
    }


def validation_score(metrics, primary=CHECKPOINT_MONITOR):
    primary = str(primary).upper()

    if primary == "AUC" and not np.isnan(metrics.get("AUC", np.nan)):
        return float(metrics["AUC"])
    if primary == "BAL_ACC":
        return float(metrics.get("BAL_ACC", 0.0))
    if primary == "ACC":
        return float(metrics.get("ACC", 0.0))
    if primary == "F1":
        return float(metrics.get("F1", 0.0))

    for fallback_name in ["AUC", "BAL_ACC", "ACC", "F1"]:
        value = metrics.get(fallback_name, np.nan)
        if not np.isnan(value):
            return float(value)

    return float("-inf")


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_prob, subjects, datasets, gate_means = [], [], [], [], []

    for batch in loader:
        mri = batch["mri"].to(DEVICE, non_blocking=True)
        pet = batch["pet"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)

        enc = model.encode(mri, pet)
        logits = model.classify(enc["fused"])
        prob = torch.softmax(logits, dim=1)[:, 1]

        y_true.extend(label.cpu().numpy().tolist())
        y_prob.extend(prob.detach().cpu().numpy().tolist())
        subjects.extend(batch["subject"])
        datasets.extend(batch["dataset"])
        gate_means.extend(enc["gate"].detach().mean(dim=1).cpu().numpy().tolist())

    return np.asarray(y_true), np.asarray(y_prob), subjects, datasets, np.asarray(gate_means)


def tune_threshold(y_true, y_prob, metric="balanced_accuracy"):
    thresholds = np.linspace(0.05, 0.95, 19)
    best_t = 0.5
    best_score = -1.0
    rows = []

    for t in thresholds:
        m = compute_metrics(y_true, y_prob, threshold=t)
        if metric == "f1":
            score = m["F1"]
        else:
            score = m["BAL_ACC"]
        rows.append({
            "threshold": float(t),
            "score": float(score),
            "BAL_ACC": m["BAL_ACC"],
            "F1": m["F1"],
            "SEN": m["SEN"],
            "SPE": m["SPE"],
        })
        if score > best_score:
            best_score = score
            best_t = float(t)

    return best_t, pd.DataFrame(rows)


@torch.no_grad()
def evaluate_model(model, loader, criterion=None, threshold=0.5):
    model.eval()

    y_true = []
    y_prob = []
    gate_means = []
    total_loss = 0.0
    n = 0

    for batch in loader:
        mri = batch["mri"].to(DEVICE, non_blocking=True)
        pet = batch["pet"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)

        enc = model.encode(mri, pet)
        logits = model.classify(enc["fused"])

        if criterion is not None:
            loss = criterion(logits, label)
            total_loss += loss.item()
            n += 1

        prob = torch.softmax(logits, dim=1)[:, 1]

        y_true.extend(label.cpu().numpy().tolist())
        y_prob.extend(prob.detach().cpu().numpy().tolist())
        gate_means.extend(enc["gate"].detach().mean(dim=1).cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_prob, threshold=threshold)
    metrics["loss"] = total_loss / max(n, 1) if criterion is not None else np.nan
    metrics["gate_mean"] = float(np.mean(gate_means)) if len(gate_means) else np.nan
    return metrics


def predictions_dataframe(model, loader, split_name, threshold=0.5):
    y_true, y_prob, subjects, datasets, gate_means = collect_predictions(model, loader)
    y_pred = (y_prob >= threshold).astype(int)
    return pd.DataFrame({
        "split": split_name,
        "subject": subjects,
        "dataset": datasets,
        "label": y_true.astype(int),
        "prob_AD": y_prob.astype(float),
        "pred": y_pred.astype(int),
        "correct": (y_pred == y_true).astype(int),
        "gate_mean": gate_means.astype(float),
    })


In [26]:

import os
import gc
import torch
import numpy as np

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Real-data GPU test device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Torch:", torch.__version__)
    print("CUDA:", torch.version.cuda)


def check_volume_stats(arr, name):
    arr = np.asarray(arr)
    print(
        name,
        "| shape:", arr.shape,
        "| finite:", np.isfinite(arr).all(),
        "| min:", float(np.nanmin(arr)),
        "| max:", float(np.nanmax(arr)),
        "| mean:", float(np.nanmean(arr)),
        "| std:", float(np.nanstd(arr)),
    )


def gpu_test_real_dataframe(df, split_name, max_items=3):
    print("\n" + "=" * 80)
    print(f"Testing split: {split_name}")
    print("=" * 80)

    model = GatedCNNFusionOnly(dropout=DROPOUT).to(DEVICE)
    model.eval()

    n = min(len(df), max_items if max_items is not None else len(df))
    for i in range(n):
        row = df.iloc[i].to_dict()
        dataset = row["dataset"]
        mri = get_cached_or_build(row, modality="MRI", index_key=f"{dataset}_MRI")
        pet = get_cached_or_build(row, modality="PET", index_key=f"{dataset}_PET")

        check_volume_stats(mri, f"{split_name} sample {i} MRI")
        check_volume_stats(pet, f"{split_name} sample {i} PET")

        x_mri = torch.tensor(mri, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        x_pet = torch.tensor(pet, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            enc = model.encode(x_mri, x_pet)
            logits = model.classify(enc["fused"])
            prob = torch.softmax(logits, dim=1)[:, 1]
            gate_mean = enc["gate"].mean(dim=1)

        print(
            "logits:", logits.detach().cpu().numpy(),
            "prob_AD:", prob.detach().cpu().numpy(),
            "gate_mean:", gate_mean.detach().cpu().numpy(),
        )

        del x_mri, x_pet, enc, logits, prob, gate_mean
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


gpu_test_real_dataframe(adni1_train_f, "ADNI1 train", max_items=1)
gpu_test_real_dataframe(adni2_train_f, "ADNI2 train", max_items=1)


Real-data GPU test device: cuda
GPU: Tesla T4
Torch: 2.10.0+cu128
CUDA: 12.8

Testing split: ADNI1 train
ADNI1 train sample 0 MRI | shape: (113, 137, 113) | finite: True | min: -0.8799219131469727 | max: 3.8825042247772217 | mean: -0.15478090941905975 | std: 0.946608304977417
ADNI1 train sample 0 PET | shape: (113, 137, 113) | finite: True | min: 0.0 | max: 0.999998927116394 | mean: 0.151887446641922 | std: 0.21188245713710785
logits: [[0.05775655 0.05894231]] prob_AD: [0.5002964] gate_mean: [0.50070894]

Testing split: ADNI2 train
ADNI2 train sample 0 MRI | shape: (113, 137, 113) | finite: True | min: -0.7957814335823059 | max: 3.2092795372009277 | mean: -0.15748712420463562 | std: 0.9482659101486206
ADNI2 train sample 0 PET | shape: (113, 137, 113) | finite: True | min: 0.0 | max: 0.9999989867210388 | mean: 0.13039295375347137 | std: 0.23329298198223114
logits: [[-0.09997752  0.00500842]] prob_AD: [0.5262224] gate_mean: [0.50039065]


In [27]:

def train_source_only(model, optimizer, source_train_loader, source_val_loader, criterion, epochs, ckpt_path):
    """Train only on labeled source-domain samples. No target data is used."""
    best_score = float("-inf")
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        n = 0

        for batch in source_train_loader:
            mri = batch["mri"].to(DEVICE, non_blocking=True)
            pet = batch["pet"].to(DEVICE, non_blocking=True)
            label = batch["label"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(mri, pet)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            n += 1

        val_metrics = evaluate_model(model, source_val_loader, criterion=criterion, threshold=0.5)
        val_score = validation_score(val_metrics)
        avg_train_loss = train_loss / max(n, 1)

        history.append({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["ACC"],
            "val_bal_acc": val_metrics["BAL_ACC"],
            "val_auc": val_metrics["AUC"],
            "val_f1": val_metrics["F1"],
            "val_gate_mean": val_metrics["gate_mean"],
            "val_score": val_score,
        })

        if val_score > best_score:
            best_score = val_score
            torch.save(model.state_dict(), ckpt_path)

        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(
                f"[Gated fusion only | source-only CE] epoch {epoch:03d}/{epochs} | "
                f"train_loss={avg_train_loss:.4f} | "
                f"val_acc={val_metrics['ACC']:.2f} | "
                f"val_bal_acc={val_metrics['BAL_ACC']:.2f} | "
                f"val_auc={val_metrics['AUC']:.2f} | "
                f"val_f1={val_metrics['F1']:.2f} | "
                f"gate_mean={val_metrics['gate_mean']:.3f} | "
                f"best_by_{CHECKPOINT_MONITOR.lower()}={best_score:.2f}"
            )

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return pd.DataFrame(history)


In [28]:

def run_transfer_experiment(source_name, target_name, source_train_df, source_val_df, source_test_df,
                            target_train_df, target_val_df, target_test_df):
    print("=" * 80)
    print(f"Running Ablation 2 gated fusion only: {source_name} -> {target_name}")
    print("=" * 80)
    print(
        f"source train/val/test = {len(source_train_df)}/{len(source_val_df)}/{len(source_test_df)} | "
        f"target train/val/test = {len(target_train_df)}/{len(target_val_df)}/{len(target_test_df)}"
    )
    print("Gated fusion is enabled. Focal loss and domain adaptation are disabled.")

    source_train_loader = make_loader(source_train_df, f"{source_name}_train", shuffle=True)
    source_val_loader   = make_loader(source_val_df,   f"{source_name}_val",   shuffle=False)
    source_test_loader  = make_loader(source_test_df,  f"{source_name}_test",  shuffle=False)

    target_val_loader   = make_loader(target_val_df,   f"{target_name}_val",   shuffle=False)
    target_test_loader  = make_loader(target_test_df,  f"{target_name}_test",  shuffle=False)

    criterion = nn.CrossEntropyLoss().to(DEVICE)
    model = GatedCNNFusionOnly(dropout=DROPOUT).to(DEVICE)

    exp_key = f"ablation2_gated_fusion_only__{source_name}_to_{target_name}"
    ckpt_path = CHECKPOINT_ROOT / f"{exp_key}__best.pt"

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    history = train_source_only(
        model=model,
        optimizer=optimizer,
        source_train_loader=source_train_loader,
        source_val_loader=source_val_loader,
        criterion=criterion,
        epochs=TRAIN_EPOCHS,
        ckpt_path=ckpt_path,
    )

    if USE_SOURCE_VAL_THRESHOLD_TUNING:
        y_val, p_val, _, _, _ = collect_predictions(model, source_val_loader)
        threshold, threshold_table = tune_threshold(y_val, p_val, metric=THRESHOLD_METRIC)
        print(f"Tuned threshold from source validation ({THRESHOLD_METRIC}): {threshold:.2f}")
    else:
        threshold = 0.5
        threshold_table = pd.DataFrame()
        print("Using fixed threshold: 0.50")

    source_test_metrics = evaluate_model(model, source_test_loader, criterion=criterion, threshold=threshold)
    target_val_metrics = evaluate_model(model, target_val_loader, criterion=criterion, threshold=threshold)
    target_test_metrics = evaluate_model(model, target_test_loader, criterion=criterion, threshold=threshold)

    pred_source_test = predictions_dataframe(model, source_test_loader, f"{exp_key}_source_test", threshold=threshold)
    pred_target_val = predictions_dataframe(model, target_val_loader, f"{exp_key}_target_val", threshold=threshold)
    pred_target_test = predictions_dataframe(model, target_test_loader, f"{exp_key}_target_test", threshold=threshold)
    predictions_all = pd.concat([pred_source_test, pred_target_val, pred_target_test], ignore_index=True)

    history_path = RESULTS_ROOT / f"{exp_key}__history.csv"
    threshold_path = RESULTS_ROOT / f"{exp_key}__threshold_table.csv"
    predictions_path = RESULTS_ROOT / f"{exp_key}__predictions.csv"

    history.to_csv(history_path, index=False)
    threshold_table.to_csv(threshold_path, index=False)
    predictions_all.to_csv(predictions_path, index=False)

    result = {
        "experiment": exp_key,
        "threshold": threshold,
        "threshold_table": threshold_table,
        "source_test": source_test_metrics,
        "target_val": target_val_metrics,
        "target_test": target_test_metrics,
        "history": history,
        "predictions": predictions_all,
        "checkpoint_path": str(ckpt_path),
        "history_path": str(history_path),
        "threshold_path": str(threshold_path),
        "predictions_path": str(predictions_path),
    }

    print("\nTarget test metrics:")
    for k in ["ACC", "BAL_ACC", "SEN", "SPE", "AUC", "F1", "TP", "TN", "FP", "FN", "gate_mean"]:
        v = target_test_metrics[k]
        print(f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}")
    print("Threshold:", threshold)
    print("Saved history:", history_path)
    print("Saved predictions:", predictions_path)

    return result


In [29]:

result_12 = run_transfer_experiment(
    source_name="ADNI1",
    target_name="ADNI2",
    source_train_df=adni1_train_f,
    source_val_df=adni1_val_f,
    source_test_df=adni1_test_f,
    target_train_df=adni2_train_f,   
    target_val_df=adni2_val_f,
    target_test_df=adni2_test_f,
)

result_21 = run_transfer_experiment(
    source_name="ADNI2",
    target_name="ADNI1",
    source_train_df=adni2_train_f,
    source_val_df=adni2_val_f,
    source_test_df=adni2_test_f,
    target_train_df=adni1_train_f,   
    target_val_df=adni1_val_f,
    target_test_df=adni1_test_f,
)


Running Ablation 2 gated fusion only: ADNI1 -> ADNI2
source train/val/test = 26/20/20 | target train/val/test = 99/75/75
Gated fusion is enabled. Focal loss and domain adaptation are disabled.
[Gated fusion only | source-only CE] epoch 001/150 | train_loss=0.7083 | val_acc=50.00 | val_bal_acc=50.00 | val_auc=40.00 | val_f1=66.67 | gate_mean=0.500 | best_by_auc=40.00
[Gated fusion only | source-only CE] epoch 010/150 | train_loss=0.6835 | val_acc=50.00 | val_bal_acc=50.00 | val_auc=57.00 | val_f1=66.67 | gate_mean=0.499 | best_by_auc=62.00
[Gated fusion only | source-only CE] epoch 020/150 | train_loss=0.7800 | val_acc=50.00 | val_bal_acc=50.00 | val_auc=57.00 | val_f1=66.67 | gate_mean=0.499 | best_by_auc=62.00
[Gated fusion only | source-only CE] epoch 030/150 | train_loss=0.7103 | val_acc=50.00 | val_bal_acc=50.00 | val_auc=49.00 | val_f1=66.67 | gate_mean=0.499 | best_by_auc=62.00
[Gated fusion only | source-only CE] epoch 040/150 | train_loss=0.7334 | val_acc=50.00 | val_bal_acc=50

In [30]:

def flatten_metrics(result, split_name):
    metrics = result[split_name]
    return {
        "experiment": result["experiment"],
        "split": split_name,
        "threshold": result["threshold"],
        "ACC": metrics["ACC"],
        "SEN": metrics["SEN"],
        "SPE": metrics["SPE"],
        "AUC": metrics["AUC"],
        "F1": metrics["F1"],
        "BAL_ACC": metrics["BAL_ACC"],
        "TP": metrics["TP"],
        "TN": metrics["TN"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "gate_mean": metrics.get("gate_mean", np.nan),
    }

summary_df = pd.DataFrame([
    flatten_metrics(result_12, "source_test"),
    flatten_metrics(result_12, "target_val"),
    flatten_metrics(result_12, "target_test"),
    flatten_metrics(result_21, "source_test"),
    flatten_metrics(result_21, "target_val"),
    flatten_metrics(result_21, "target_test"),
])

summary_path = RESULTS_ROOT / "ablation2_gated_fusion_only_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)
display(summary_df)

paper_table_rows = summary_df[summary_df["split"] == "target_test"].copy()
paper_table_rows["Method"] = "Gated fusion only"
paper_table_rows["Gated fusion"] = "Yes"
paper_table_rows["Focal loss"] = "No"
paper_table_rows["Domain adaptation"] = "No"
paper_table_rows = paper_table_rows[[
    "experiment", "Method", "Gated fusion", "Focal loss", "Domain adaptation",
    "ACC", "SEN", "SPE", "AUC", "F1", "gate_mean"
]]

paper_table_path = RESULTS_ROOT / "ablation2_gated_fusion_only_paper_rows.csv"
paper_table_rows.to_csv(paper_table_path, index=False)
print("Saved paper-table rows:", paper_table_path)
display(paper_table_rows)


Saved summary: /kaggle/working/ablation2_gated_fusion_only/results_ablation2_gated_fusion/ablation2_gated_fusion_only_summary.csv


,experiment,split,threshold,ACC,SEN,SPE,AUC,F1,BAL_ACC,TP,TN,FP,FN,gate_mean
0,ablation2_gated_fusion_only__ADNI1_to_ADNI2,source_test,0.05,50.000000,100.0,0.0,39.00,66.666667,50.0,10,0,10,0,0.498967
1,ablation2_gated_fusion_only__ADNI1_to_ADNI2,target_val,0.05,33.333333,100.0,0.0,55.84,50.000000,50.0,25,0,50,0,0.498125
2,ablation2_gated_fusion_only__ADNI1_to_ADNI2,target_test,0.05,33.333333,100.0,0.0,53.84,50.000000,50.0,25,0,50,0,0.498500
3,ablation2_gated_fusion_only__ADNI2_to_ADNI1,source_test,0.45,56.000000,64.0,52.0,61.04,49.230769,58.0,16,26,24,9,0.499278
4,ablation2_gated_fusion_only__ADNI2_to_ADNI1,target_val,0.45,45.000000,20.0,70.0,46.00,26.666667,45.0,2,7,3,8,0.500581
5,ablation2_gated_fusion_only__ADNI2_to_ADNI1,target_test,0.45,40.000000,20.0,60.0,50.00,25.000000,40.0,2,6,4,8,0.500662


Saved paper-table rows: /kaggle/working/ablation2_gated_fusion_only/results_ablation2_gated_fusion/ablation2_gated_fusion_only_paper_rows.csv


,experiment,Method,Gated fusion,Focal loss,Domain adaptation,ACC,SEN,SPE,AUC,F1,gate_mean
2,ablation2_gated_fusion_only__ADNI1_to_ADNI2,Gated fusion only,Yes,No,No,33.333333,100.0,0.0,53.84,50.0,0.498500
5,ablation2_gated_fusion_only__ADNI2_to_ADNI1,Gated fusion only,Yes,No,No,40.000000,20.0,60.0,50.00,25.0,0.500662
